In [48]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
textos = [
    "oi olá tudo bem END",
    "tudo bem estou bem obrigado END",
    "qual seu nome sou um chatbot simples END",
    "quem te criou fui criado com tensorflow END",
    "bom dia bom dia para você END",
    "boa noite boa noite END",
    "como você está estou funcionando bem END",
    "o que você faz eu converso com você END",
    "até mais até logo END"
]


tokenizer = Tokenizer(filters='')
tokenizer.fit_on_texts(textos)

total_words = len(tokenizer.word_index) + 1


input_sequences = []

for texto in textos:
    token_list = tokenizer.texts_to_sequences([texto])[0]

    for i in range(1, len(token_list)):
        seq = token_list[:i + 1]
        input_sequences.append(seq)

max_sequence_len = max(len(seq) for seq in input_sequences)

input_sequences = np.array(
    pad_sequences(
        input_sequences,
        maxlen=max_sequence_len,
        padding='pre'
    )
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
    
y = tf.keras.utils.to_categorical(
    y,
    num_classes=total_words
)

In [50]:
model = Sequential([
    Embedding(total_words, 64, input_length=max_sequence_len - 1),
    LSTM(128),
    Dense(total_words, activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.build(input_shape=(None, max_sequence_len - 1))
model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_11 (Embedding)        │ (None, 8, 64)          │         2,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 39)             │         5,031 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 106,343 (415.40 KB)

 Trainable params: 106,343 (415.40 KB)

 Non-trainable params: 0 (0.00 B)

In [51]:

model.fit(
    X,
    y,
    epochs=500,
    verbose=1
)

Epoch 1/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 54ms/step - accuracy: 0.0392 - loss: 3.6633
Epoch 2/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.1961 - loss: 3.6506
Epoch 3/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.2157 - loss: 3.6394
Epoch 4/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.2157 - loss: 3.6243
Epoch 5/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.1961 - loss: 3.6072
Epoch 6/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.1765 - loss: 3.5875
Epoch 7/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.1765 - loss: 3.5584
Epoch 8/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.1765 - loss: 3.5183
Epoch 9/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.1765 - loss: 3.4574
Epoch 10/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.1765 - loss: 3.3976
Epoch 11/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.1765 - loss: 3.3440
Epoch 12/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.1765 - lo

In [ ]:

def gerar_resposta(seed_text, max_words=20):
    resposta = seed_text

    for _ in range(max_words):
        token_list = tokenizer.texts_to_sequences([resposta])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_len - 1,
            padding='pre'
        )

        predicted = model.predict(
            token_list,
            verbose=0
        )

        predicted_index = np.argmax(predicted)

        next_word = ""

        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                next_word = word
                break

        if next_word == "end":
            break

        resposta += " " + next_word

    return resposta


In [57]:
a = input("Diga um numero")
print(int(a)+1)

3


In [61]:

print("\nChat iniciado! Digite 'sair'\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print(" ")
        break
    print(entrada)
    resposta = gerar_resposta(entrada)
    print("Chatbot:", resposta)


Chat iniciado! Digite 'sair'

oi
Chatbot: oi olá tudo bem
 


In [65]:
import numpy as np

# =====================================================
# Configuração do ponto fixo
#
# Exemplo:
# Q8.8  -> 16 bits total
#         8 bits parte inteira
#         8 bits parte fracionária
#
# escala = 2^fractional_bits
# =====================================================

TOTAL_BITS = 16
FRACTIONAL_BITS = 8
SCALE = 2 ** FRACTIONAL_BITS

# =====================================================
# Função float -> binário em ponto fixo
# =====================================================

def float_to_fixed_bin(value,
                       total_bits=TOTAL_BITS,
                       fractional_bits=FRACTIONAL_BITS):

    # converte para inteiro escalado
    fixed_value = int(round(value * (2 ** fractional_bits)))

    # complemento de 2 para negativos
    if fixed_value < 0:
        fixed_value = (1 << total_bits) + fixed_value

    # máscara de segurança
    fixed_value = fixed_value & ((1 << total_bits) - 1)

    # retorna string binária
    return format(fixed_value, f'0{total_bits}b')

# =====================================================
# Percorrer todas as camadas e converter pesos
# =====================================================

for i, layer in enumerate(model.layers):
    print(f"\n===================================")
    print(f"Camada {i}: {layer.name}")
    print(f"Tipo: {type(layer).__name__}")

    pesos = layer.get_weights()

    if len(pesos) == 0:
        print("Sem pesos treináveis.")
        continue

    for j, matriz in enumerate(pesos):
        print(f"\nPeso {j}")
        print(f"Shape: {matriz.shape}")

        flat = matriz.flatten()

        print("\nPrimeiros 20 valores convertidos:")

        for valor in flat[:20]:
            binario = float_to_fixed_bin(valor)
            print(f"{valor: .6f} -> {binario}")


Camada 0: embedding_11
Tipo: Embedding

Peso 0
Shape: (39, 64)

Primeiros 20 valores convertidos:
 0.093753 -> 0000000000011000
-0.056515 -> 1111111111110010
-0.011717 -> 1111111111111101
 0.065608 -> 0000000000010001
-0.110421 -> 1111111111100100
 0.027924 -> 0000000000000111
-0.168580 -> 1111111111010101
 0.064780 -> 0000000000010001
 0.158128 -> 0000000000101000
-0.137227 -> 1111111111011101
 0.077247 -> 0000000000010100
-0.087702 -> 1111111111101010
-0.137581 -> 1111111111011101
 0.022855 -> 0000000000000110
-0.104810 -> 1111111111100101
-0.006646 -> 1111111111111110
-0.113068 -> 1111111111100011
 0.044540 -> 0000000000001011
-0.157018 -> 1111111111011000
-0.219715 -> 1111111111001000

Camada 1: lstm_11
Tipo: LSTM

Peso 0
Shape: (64, 512)

Primeiros 20 valores convertidos:
 0.095206 -> 0000000000011000
 0.046761 -> 0000000000001100
 0.065468 -> 0000000000010001
 0.107924 -> 0000000000011100
 0.138832 -> 0000000000100100
 0.088280 -> 0000000000010111
 0.083992 -> 0000000000010110
 